# thaw quickstart

Fast snapshot/restore for LLM inference. This notebook demos the full flow:
1. `pip install thaw-vllm`
2. Load a model normally and **freeze** weights to a snapshot
3. **Restore** from snapshot with dummy init (skip weight download)
4. Run inference — identical output

**Requirements:** Colab with GPU runtime (T4 is fine).

In [ ]:
# Install thaw-vllm + fix Colab compatibility
!pip install git+https://github.com/thaw-ai/thaw.git -q
!pip install vllm fastapi uvicorn -q
!pip install "sympy==1.13.3" -q

**Restart the runtime now** (Runtime > Restart session), then run the cells below.

In [ ]:
# Colab/Jupyter compatibility patches
import sys, os
sys.stdout._original_stdstream_copy = 1
sys.stderr._original_stdstream_copy = 2
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

## Step 1: Freeze — load model and snapshot weights

In [ ]:
import time
from vllm import LLM, SamplingParams
from thaw_vllm import freeze_model_pipelined

# Normal model load
print("Loading model normally...")
t0 = time.perf_counter()
llm = LLM(model="facebook/opt-125m", dtype="float16", enforce_eager=True)
normal_load = time.perf_counter() - t0
print(f"Normal load: {normal_load:.1f}s")

# Get model reference
core = llm.llm_engine.engine_core
if hasattr(core, 'engine_core'):
    core = core.engine_core
model = core.model_executor.driver_worker.model_runner.model

# Freeze weights to snapshot
stats = freeze_model_pipelined(model, "/content/weights.thaw")
print(f"Frozen: {stats['num_regions']} regions, {stats['total_bytes']/1e6:.0f} MB in {stats['elapsed_s']:.2f}s")

# Verify inference works
out = llm.generate(["The capital of France is"], SamplingParams(temperature=0, max_tokens=20))
original_output = out[0].outputs[0].text
print(f"Original output: {original_output}")

# Clean up GPU memory
del llm, model, core
import torch, gc
gc.collect()
torch.cuda.empty_cache()

## Step 2: Restore — dummy init + thaw snapshot

In [ ]:
from thaw_vllm import restore_model_pipelined

# Fast init with dummy weights (no download, no deserialization)
print("Initializing with dummy weights...")
t0 = time.perf_counter()
llm2 = LLM(model="facebook/opt-125m", dtype="float16", enforce_eager=True, load_format="dummy")
init_time = time.perf_counter() - t0
print(f"Dummy init: {init_time:.1f}s")

# Restore real weights from snapshot
core2 = llm2.llm_engine.engine_core
if hasattr(core2, 'engine_core'):
    core2 = core2.engine_core
model2 = core2.model_executor.driver_worker.model_runner.model

t0 = time.perf_counter()
rstats = restore_model_pipelined(model2, "/content/weights.thaw")
restore_time = time.perf_counter() - t0
thaw_total = init_time + restore_time

print(f"Restore: {restore_time:.2f}s ({rstats['throughput_gb_s']:.2f} GB/s)")
print(f"\nTotal: {thaw_total:.1f}s vs normal {normal_load:.1f}s = {normal_load/thaw_total:.1f}x faster")

## Step 3: Verify — identical output

In [ ]:
out2 = llm2.generate(["The capital of France is"], SamplingParams(temperature=0, max_tokens=20))
restored_output = out2[0].outputs[0].text

print(f"Original: {original_output}")
print(f"Restored: {restored_output}")
print(f"\nBit-identical: {'PASS' if original_output == restored_output else 'FAIL'}")